In [1]:
%run functions.ipynb

import os

os.makedirs("../submission", exist_ok=True)

df = load_data()
df.head()

,yt_true
date,
1946-01-01,890
1946-02-01,992
1946-03-01,979
1946-04-01,959
1946-05-01,1110


In [2]:
# Model 6: MLP autoregressive model over the scaled series
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import MinMaxScaler

P = 13
scaler = MinMaxScaler()
df = make_yt_true_scaled(df, scaler)
df = make_lagged_ts(df, P, "yt_true_scaled")
x_columns = [f"lagged_{i}m" for i in range(1, P + 1)]

data = df.dropna()
X_complete, y_complete, X_train, y_train, X_test, y_test = train_test_split(
    data, x_columns, "yt_true_scaled"
)

mlp = MLPRegressor(
    hidden_layer_sizes=(8,),
    activation="logistic",
    solver="lbfgs",
    max_iter=10000,
    random_state=42,
)
mlp.fit(X_train, y_train)

df["yt_pred_mlp"] = np.nan
df.loc[data.index, "yt_pred_mlp"] = scaler.inverse_transform(
    mlp.predict(X_complete).reshape(-1, 1)
).ravel()
plot_time_series(df[["yt_true", "yt_pred_mlp"]])

<Figure size 1200x400 with 1 Axes>

In [3]:
metrics = compute_evaluation_metrics(df[["yt_true", "yt_pred_mlp"]].dropna())
save_forecasts(df)
save_metrics(metrics)
metrics

,Metrics,yt_pred_mlp
0,MSE Train,77476.62
1,MSE Test,128494.15
2,MAE Train,201.12
3,MAE Test,266.03
